In [ ]:

import os
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc, roc_auc_score, r2_score
import cv2
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier
from skimage.feature import graycomatrix, graycoprops
from keras.applications.densenet import DenseNet121
from keras.preprocessing import image
from keras.applications.densenet import preprocess_input
from tensorflow.keras.models import Model

In [2]:
# Try loading the CSV file with a different encoding
soil_report = pd.read_csv('/kaggle/input/soil-fertility-prediction-dataset/soil_fertility_dataset/soil_test_report.csv', encoding='ISO-8859-1')

# Display the first few rows of the soil report
soil_report.head()


# The directory for the microscopic soil images
IMAGE_DIR = '/kaggle/input/soil-fertility-prediction-dataset/soil_fertility_dataset/Microscopic_soil_image'


In [ ]:
import os
import numpy as np
import pandas as pd
import cv2
from skimage.feature import graycomatrix, graycoprops

# SET PATHS + SEED (one place)

SEED = 42
np.random.seed(SEED)

IMAGE_DIR = "/kaggle/input/soil-fertility-prediction-dataset/soil_fertility_dataset/Microscopic_soil_image"

# 1) Helper: list image files

def list_image_files(image_dir, exts=(".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp")):
    files = []
    for fn in os.listdir(image_dir):
        if fn.lower().endswith(exts):
            files.append(fn)
    files.sort()
    return files

# GLCM extractor function
# Features: contrast, dissimilarity, homogeneity, energy, correlation, ASM

def glcm_features_from_path(
    image_path,
    resize_to=None,          # e.g. (256, 256) or None
    distances=(1,),          
    angles=(0, np.pi/4, np.pi/2, 3*np.pi/4),
    levels=256               
):
    # Read grayscale (0-255)
    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        raise ValueError(f"Could not read image: {image_path}")

    # Optional resize (helps speed + consistency)
    if resize_to is not None:
        img = cv2.resize(img, resize_to, interpolation=cv2.INTER_AREA)

    # Ensure uint8 and in [0, levels-1]
    if img.dtype != np.uint8:
        img = np.clip(img, 0, 255).astype(np.uint8)

    # Compute GLCM
    glcm = graycomatrix(
        img,
        distances=distances,
        angles=angles,
        levels=levels,
        symmetric=True,
        normed=True
    )

    # Extract props (mean over distances and angles)
    feats = {
        "contrast": float(graycoprops(glcm, "contrast").mean()),
        "dissimilarity": float(graycoprops(glcm, "dissimilarity").mean()),
        "homogeneity": float(graycoprops(glcm, "homogeneity").mean()),
        "energy": float(graycoprops(glcm, "energy").mean()),
        "correlation": float(graycoprops(glcm, "correlation").mean()),
        "ASM": float(graycoprops(glcm, "ASM").mean()),
    }
    return feats

# Run extraction for all images and build the exact table you want.....

image_files = list_image_files(IMAGE_DIR)

rows = []
for fn in image_files:
    path = os.path.join(IMAGE_DIR, fn)
    feats = glcm_features_from_path(
        path,
        resize_to=None,          
        distances=(1,),
        angles=(0, np.pi/4, np.pi/2, 3*np.pi/4),
        levels=256
    )
    feats["image"] = fn
    rows.append(feats)

#  DataFrame with the extracted features
glcm_df = pd.DataFrame(rows, columns=["contrast","dissimilarity","homogeneity","energy","correlation","ASM","image"])


# SHOW like your example

print(glcm_df.head(10))


#  Save  CSV File

glcm_df.to_csv('/kaggle/working/glcm_features.csv', index=False)
print("GLCM features saved to 'glcm_features.csv'.")


In [ ]:
import os
import numpy as np
import pandas as pd
import cv2
from skimage.feature import graycomatrix, graycoprops
from sklearn.preprocessing import StandardScaler


# SET PATHS + SEED (one place)

SEED = 42
np.random.seed(SEED)

IMAGE_DIR = "/kaggle/input/soil-fertility-prediction-dataset/soil_fertility_dataset/Microscopic_soil_image"
CSV_PATH = "/kaggle/input/soil-fertility-prediction-dataset/soil_fertility_dataset/soil_test_report.csv"


# Helper: list image files

def list_image_files(image_dir, exts=(".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp")):
    files = []
    for fn in os.listdir(image_dir):
        if fn.lower().endswith(exts):
            files.append(fn)
    files.sort()
    return files

# GLCM extractor function
# Features: contrast, dissimilarity, homogeneity, energy, correlation, ASM

def glcm_features_from_path(
    image_path,
    resize_to=None,          # e.g. (256, 256) or None
    distances=(1,),          # you can add more distances like (1,2,3)
    angles=(0, np.pi/4, np.pi/2, 3*np.pi/4),
    levels=256               # keep 256 for uint8 grayscale
):
    # Read grayscale (0-255)
    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        raise ValueError(f"Could not read image: {image_path}")

    # resize (helps speed + consistency)
    if resize_to is not None:
        img = cv2.resize(img, resize_to, interpolation=cv2.INTER_AREA)

    # Ensure uint8 and in [0, levels-1]
    if img.dtype != np.uint8:
        img = np.clip(img, 0, 255).astype(np.uint8)

    # Compute GLCM
    glcm = graycomatrix(
        img,
        distances=distances,
        angles=angles,
        levels=levels,
        symmetric=True,
        normed=True
    )

    # Extract props (mean over distances and angles)
    feats = {
        "contrast": float(graycoprops(glcm, "contrast").mean()),
        "dissimilarity": float(graycoprops(glcm, "dissimilarity").mean()),
        "homogeneity": float(graycoprops(glcm, "homogeneity").mean()),
        "energy": float(graycoprops(glcm, "energy").mean()),
        "correlation": float(graycoprops(glcm, "correlation").mean()),
        "ASM": float(graycoprops(glcm, "ASM").mean()),
    }
    return feats


# Load soil data and clean image names
soil_df = pd.read_csv(CSV_PATH, encoding="latin1")

# Selecting required columns and renaming
soil_df = soil_df[["Image", "Fertility"]]
soil_df.columns = ["image", "fertility"]

# Remove rows with missing values
soil_df = soil_df.dropna(subset=["image", "fertility"])

# Clean image filenames: strip spaces and add '.jpg' if missing
soil_df["image"] = soil_df["image"].astype(str).str.strip()
soil_df["image"] = soil_df["image"].apply(
    lambda x: x + ".jpg" if not x.lower().endswith(".jpg") else x
)

# Filter images based on the selected files in IMAGE_DIR
selected_images = set(os.listdir(IMAGE_DIR))
soil_df = soil_df[soil_df["image"].isin(selected_images)]

# Reset index
soil_df = soil_df.reset_index(drop=True)


#Extract GLCM features for all images

image_files = list_image_files(IMAGE_DIR)

# Prepare a list for the feature extraction
rows = []
for fn in image_files:
    path = os.path.join(IMAGE_DIR, fn)
    feats = glcm_features_from_path(
        path,
        resize_to=None,  
        distances=(1,),
        angles=(0, np.pi/4, np.pi/2, 3*np.pi/4),
        levels=256
    )
    feats["image"] = fn  # Retain the 'image' column
    rows.append(feats)

# Create a DataFrame with the extracted features
glcm_df = pd.DataFrame(rows, columns=["contrast", "dissimilarity", "homogeneity", "energy", "correlation", "ASM", "image"])


# Standardize the GLCM Features

# Extract features (excluding the 'image' column) for standardization
features = glcm_df[['contrast', 'dissimilarity', 'homogeneity', 'energy', 'correlation', 'ASM']]

# Apply StandardScaler to standardize the features
scaler = StandardScaler()
scaled_features = scaler.fit_transform(features)

# Convert the standardized features back into a DataFrame
scaled_glcm_df = pd.DataFrame(scaled_features, columns=['contrast', 'dissimilarity', 'homogeneity', 'energy', 'correlation', 'ASM'])

# Add the 'image' column back to the standardized DataFrame
scaled_glcm_df['image'] = glcm_df['image']


# Merge with fertility labels

# Merge the standardized features with the fertility labels from soil_df
final_df = pd.merge(scaled_glcm_df, soil_df[['image', 'fertility']], on='image', how='left')

# Check for NaN values in the fertility column after merge
missing_fertility_count = final_df['fertility'].isna().sum()
print(f"Number of rows with missing fertility labels after merge: {missing_fertility_count}")

# If there are still missing fertility labels, print those rows
if missing_fertility_count > 0:
    print("Rows with missing fertility labels:")
    print(final_df[final_df['fertility'].isna()][['image', 'fertility']])

# Display the final DataFrame
print(final_df.head())


# Save the final DataFrame to CSV

final_df.to_csv('/kaggle/working/glcm_standardized_features_with_fertility.csv', index=False)
print("Final DataFrame with GLCM features and fertility labels saved to '/kaggle/working/glcm_standardized_features_with_fertility.csv'")




In [ ]:

import numpy as np
import pandas as pd
import os
import cv2
from skimage import io
from sklearn.preprocessing import StandardScaler


# Optimized GLRM feature extractor

def glrm_features_from_path(image_path, resize_to=None):
    """
    Extracts Gray-Level Run-Length Matrix (GLRM) features from the image.
    
    Features:
    - Short Run Emphasis (SRE)
    - Long Run Emphasis (LRE)
    - Gray Level Nonuniformity (GLN)
    - Run Percentage (RP)
    
    Args:
    - image_path (str): path to the image file
    - resize_to (tuple): optional resizing of the image (height, width)
    
    Returns:
    - dict: features extracted from the image
    """
    # Read image in grayscale
    img = io.imread(image_path, as_gray=True)
    img = (img * 255).astype(np.uint8)  # Convert to 8-bit grayscale (0-255)
    
    # Resize image if needed
    if resize_to is not None:
        img = cv2.resize(img, resize_to, interpolation=cv2.INTER_AREA)

    gray_levels = np.unique(img)  # unique gray levels
    num_gray_levels = len(gray_levels)
    rows, cols = img.shape

    # Initialize a matrix to count runs
    run_length_matrix = np.zeros((num_gray_levels, rows), dtype=int)

    # Vectorized calculation of runs for each gray level
    for i, gray_level in enumerate(gray_levels):
        # Create a mask for each gray level
        mask = (img == gray_level)

        # For each row, calculate run lengths
        for row_idx in range(rows):
            row = mask[row_idx]
            run_length = 0
            for col_idx in range(cols):
                if row[col_idx] == 1:
                    run_length += 1
                elif run_length > 0:
                    run_length_matrix[i, run_length - 1] += 1
                    run_length = 0

            # Add final run if it ended at the last column
            if run_length > 0:
                run_length_matrix[i, run_length - 1] += 1

    # Extract features based on the run-length matrix
    run_length_sums = np.sum(run_length_matrix, axis=0)
    total_runs = np.sum(run_length_sums)

    # Short Run Emphasis (SRE)
    SRE = np.sum(run_length_sums ** 2) / total_runs if total_runs != 0 else 0

    # Long Run Emphasis (LRE)
    LRE = np.sum(run_length_sums / (np.arange(1, run_length_sums.shape[0] + 1) ** 2)) / total_runs if total_runs != 0 else 0

    # Gray Level Nonuniformity (GLN)
    GLN = np.sum(run_length_sums ** 2) / (rows ** 2) if rows > 0 else 0

    # Run Percentage (RP)
    RP = total_runs / (rows * cols) if rows * cols > 0 else 0

    return {
        "SRE": SRE,
        "LRE": LRE,
        "GLN": GLN,
        "RP": RP
    }


# Efficiently extract GLRM features for all images

def extract_glrm_for_all_images(image_dir, resize_to=None):
    image_files = [f for f in os.listdir(image_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff'))]
    image_files.sort()

    rows = []
    for fn in image_files:
        path = os.path.join(image_dir, fn)
        features = glrm_features_from_path(path, resize_to=resize_to)  # Resize if needed (e.g., (256, 256))
        features["image"] = fn
        rows.append(features)

    glrm_df = pd.DataFrame(rows, columns=["SRE", "LRE", "GLN", "RP", "image"])
    
    return glrm_df

#
#  Extract GLRM features and display first few rows

GLRM_DATAFRAME = extract_glrm_for_all_images("/kaggle/input/soil-fertility-prediction-dataset/soil_fertility_dataset/Microscopic_soil_image", resize_to=(256, 256))

# Display first 10 rows of extracted GLRM features
print(GLRM_DATAFRAME.head(10))


# Standardize the GLRM Features

# Extract features (excluding the 'image' column) for standardization
features = GLRM_DATAFRAME[['SRE', 'LRE', 'GLN', 'RP']]

# Apply StandardScaler to standardize the features
scaler = StandardScaler()
scaled_features = scaler.fit_transform(features)

# Convert the standardized features back into a DataFrame
scaled_glrm_df = pd.DataFrame(scaled_features, columns=['SRE', 'LRE', 'GLN', 'RP'])

# Add the 'image' column back to the standardized DataFrame
scaled_glrm_df['image'] = GLRM_DATAFRAME['image']


# Load the fertility labels and clean the data

soil_df = pd.read_csv('/kaggle/input/soil-fertility-prediction-dataset/soil_fertility_dataset/soil_test_report.csv', encoding='latin1')

soil_df = soil_df[["Image", "Fertility"]]
soil_df.columns = ["image", "fertility"]
soil_df = soil_df.dropna(subset=["image", "fertility"])

# Clean image filenames
soil_df["image"] = soil_df["image"].astype(str).str.strip()

# Add .jpg if missing
soil_df["image"] = soil_df["image"].apply(
    lambda x: x + ".jpg" if not x.lower().endswith(".jpg") else x
)

# Filter using selected images
selected_images = set(os.listdir("/kaggle/input/soil-fertility-prediction-dataset/soil_fertility_dataset/Microscopic_soil_image"))
soil_df = soil_df[soil_df["image"].isin(selected_images)]

soil_df = soil_df.reset_index(drop=True)


# Merge the GLRM features with fertility labels

final_df = pd.merge(scaled_glrm_df, soil_df[['image', 'fertility']], on='image', how='left')

# Check for NaN values in the fertility column after merge
missing_fertility_count = final_df['fertility'].isna().sum()
print(f"Number of rows with missing fertility labels after merge: {missing_fertility_count}")

# If there are still missing fertility labels, print those rows
if missing_fertility_count > 0:
    print("Rows with missing fertility labels:")
    print(final_df[final_df['fertility'].isna()][['image', 'fertility']])

# Display the final DataFrame
print(final_df.head())


#Save the final DataFrame to CSV

final_df.to_csv('/kaggle/working/glrm_standardized_features_with_fertility.csv', index=False)
print("Final DataFrame with GLRM features and fertility labels saved to 'glrm_features_with_fertility.csv'.")


In [ ]:
# dense
import os
import numpy as np
import pandas as pd
from tensorflow.keras.applications import DenseNet121
from tensorflow.keras.applications.densenet import preprocess_input
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras.models import Model
from tensorflow.keras.layers import GlobalAveragePooling2D
from sklearn.preprocessing import StandardScaler


IMAGE_DIR = "/kaggle/input/soil-fertility-prediction-dataset/soil_fertility_dataset/Microscopic_soil_image"
CSV_PATH = "/kaggle/input/soil-fertility-prediction-dataset/soil_fertility_dataset/soil_test_report.csv"

WEIGHTS_PATH = (
    "/kaggle/input/densnet121-weights/densenet121_weights_tf_dim_ordering_tf_kernels_notop.h5"
)


# LOAD DENSENET121 (ONCE)

base_model = DenseNet121(
    weights=WEIGHTS_PATH,
    include_top=False,
    input_shape=(224, 224, 3)
)

x = GlobalAveragePooling2D()(base_model.output)
model = Model(inputs=base_model.input, outputs=x)

print(" DenseNet121 loaded with ImageNet weights")


#FEATURE EXTRACTOR

def extract_densenet_features(image_path, target_size=(224, 224)):
    img = load_img(image_path, target_size=target_size)
    img = img_to_array(img)
    img = np.expand_dims(img, axis=0)
    img = preprocess_input(img)

    features = model.predict(img, verbose=0)
    return features.flatten()   # (1024,)


# EXTRACT FEATURES FOR ALL IMAGES

image_files = sorted([
    f for f in os.listdir(IMAGE_DIR)
    if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff'))
])

rows = []
for fn in image_files:
    path = os.path.join(IMAGE_DIR, fn)
    feats = extract_densenet_features(path)

    row = {f"densenet_{i+1}": feats[i] for i in range(len(feats))}
    row["image"] = fn
    rows.append(row)

densenet_df = pd.DataFrame(rows)
print("DenseNet feature shape:", densenet_df.shape)
print(densenet_df.head())


# LOAD & CLEAN FERTILITY LABELS

soil_df = pd.read_csv(CSV_PATH, encoding="latin1")

soil_df = soil_df[["Image", "Fertility"]]
soil_df.columns = ["image", "fertility"]
soil_df = soil_df.dropna(subset=["image", "fertility"])

soil_df["image"] = soil_df["image"].astype(str).str.strip()
soil_df["image"] = soil_df["image"].apply(
    lambda x: x if x.lower().endswith(".jpg") else x + ".jpg"
)

soil_df = soil_df[soil_df["image"].isin(image_files)].reset_index(drop=True)


# STANDARDIZE DENSENET FEATURES

feature_cols = [c for c in densenet_df.columns if c.startswith("densenet_")]

scaler = StandardScaler()
densenet_df[feature_cols] = scaler.fit_transform(densenet_df[feature_cols])

print(" DenseNet features standardized")


# MERGE FEATURES + LABELS

final_df = pd.merge(
    densenet_df,
    soil_df,
    on="image",
    how="left"
)

missing = final_df["fertility"].isna().sum()
print(f"Missing fertility labels after merge: {missing}")

print(final_df.head())


# SAVE FINAL CSV

final_df.to_csv(
    "/kaggle/working/densenet121_standardized_features_with_fertility.csv",
    index=False
)

print(" Saved: densenet121_standardized_features_with_fertility.csv")


In [ ]:
#combine glcm+ glrm
import pandas as pd


#LOAD STANDARDIZED FEATURE FILES

glcm_path = "/kaggle/working/glcm_standardized_features_with_fertility.csv"
glrm_path = "/kaggle/working/glrm_standardized_features_with_fertility.csv"

glcm_df = pd.read_csv(glcm_path)
glrm_df = pd.read_csv(glrm_path)

print("GLCM shape:", glcm_df.shape)
print("GLRM shape:", glrm_df.shape)


# CHECK REQUIRED COLUMNS

assert "image" in glcm_df.columns
assert "image" in glrm_df.columns
assert "fertility" in glcm_df.columns
assert "fertility" in glrm_df.columns


# DROP DUPLICATE FERTILITY FROM ONE SIDE

glrm_df = glrm_df.drop(columns=["fertility"])


# MERGE (FEATURE FUSION)

combined_df = pd.merge(
    glcm_df,
    glrm_df,
    on="image",
    how="inner"   # inner ensures only common images
)

print("Combined feature shape:", combined_df.shape)
print(combined_df.head())


# SAVE COMBINED FEATURES

output_path = "/kaggle/working/glcm_glrm_combined_features_with_fertility.csv"
combined_df.to_csv(output_path, index=False)

print(f" Combined GLCM + GLRM features saved to:\n{output_path}")


In [ ]:
#GLCM+Densenet
import pandas as pd

#  LOAD STANDARDIZED FEATURE FILES

glcm_path = "/kaggle/working/glcm_standardized_features_with_fertility.csv"
densenet_path = "/kaggle/working/densenet121_standardized_features_with_fertility.csv"

glcm_df = pd.read_csv(glcm_path)
densenet_df = pd.read_csv(densenet_path)

print("GLCM shape:", glcm_df.shape)
print("DenseNet shape:", densenet_df.shape)


# CHECK REQUIRED COLUMNS

assert "image" in glcm_df.columns
assert "image" in densenet_df.columns
assert "fertility" in glcm_df.columns
assert "fertility" in densenet_df.columns


# DROP DUPLICATE FERTILITY COLUMN

densenet_df = densenet_df.drop(columns=["fertility"])


#  MERGE (FEATURE FUSION)

combined_df = pd.merge(
    glcm_df,
    densenet_df,
    on="image",
    how="inner"   # keep only images present in both
)

print("Combined GLCM + DenseNet shape:", combined_df.shape)
print(combined_df.head())


# SAVE COMBINED FEATURES

output_path = "/kaggle/working/glcm_densenet_combined_features_with_fertility.csv"
combined_df.to_csv(output_path, index=False)

print(f" Combined GLCM + DenseNet features saved to:\n{output_path}")


In [ ]:
#glrm+ Densnet

import pandas as pd


# LOAD STANDARDIZED FEATURE FILES

glrm_path = "/kaggle/working/glrm_standardized_features_with_fertility.csv"
densenet_path = "/kaggle/working/densenet121_standardized_features_with_fertility.csv"

glrm_df = pd.read_csv(glrm_path)
densenet_df = pd.read_csv(densenet_path)

print("GLRM shape:", glrm_df.shape)
print("DenseNet shape:", densenet_df.shape)


# CHECK REQUIRED COLUMNS

assert "image" in glrm_df.columns
assert "image" in densenet_df.columns
assert "fertility" in glrm_df.columns
assert "fertility" in densenet_df.columns


# DROP DUPLICATE FERTILITY COLUMN

densenet_df = densenet_df.drop(columns=["fertility"])


# MERGE (FEATURE FUSION)

combined_df = pd.merge(
    glrm_df,
    densenet_df,
    on="image",
    how="inner"   
)

print("Combined GLRM + DenseNet shape:", combined_df.shape)
print(combined_df.head())


# SAVE COMBINED FEATURES

output_path = "/kaggle/working/glrm_densenet_combined_features_with_fertility.csv"
combined_df.to_csv(output_path, index=False)

print(f"Combined GLRM + DenseNet features saved to:\n{output_path}")


In [ ]:
#glcm+glrm+densenet

import pandas as pd


# 1) LOAD STANDARDIZED FEATURE FILES
glcm_path = "/kaggle/working/glcm_standardized_features_with_fertility.csv"
glrm_path = "/kaggle/working/glrm_standardized_features_with_fertility.csv"
densenet_path = "/kaggle/working/densenet121_standardized_features_with_fertility.csv"

glcm_df = pd.read_csv(glcm_path)
glrm_df = pd.read_csv(glrm_path)
densenet_df = pd.read_csv(densenet_path)

print("GLCM shape:", glcm_df.shape)
print("GLRM shape:", glrm_df.shape)
print("DenseNet shape:", densenet_df.shape)


# CHECK REQUIRED COLUMNS

assert "image" in glcm_df.columns
assert "image" in glrm_df.columns
assert "image" in densenet_df.columns

assert "fertility" in glcm_df.columns
assert "fertility" in glrm_df.columns
assert "fertility" in densenet_df.columns


# DROP DUPLICATE FERTILITY COLUMNS

glrm_df = glrm_df.drop(columns=["fertility"])
densenet_df = densenet_df.drop(columns=["fertility"])


#  MERGE ALL FEATURES (FEATURE-LEVEL FUSION)

combined_df = glcm_df.merge(glrm_df, on="image", how="inner") \
                      .merge(densenet_df, on="image", how="inner")

print("Combined GLCM + GLRM + DenseNet shape:", combined_df.shape)
print(combined_df.head())


# SAVE FINAL COMBINED DATASET

output_path = "/kaggle/working/F_glcm_glrm_densenet_combined_features_with_fertility.csv"
combined_df.to_csv(output_path, index=False)

print(f" Combined GLCM + GLRM + DenseNet features saved to:\n{output_path}")
